In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database")

print("Path to dataset files:", path)

100%|██████████| 26.4M/26.4M [00:01<00:00, 16.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/kaustubhdikshit/neu-surface-defect-database/versions/1


In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50, InceptionV3, EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from sklearn.metrics import classification_report
import numpy as np

In [ ]:
data_dir = os.path.join(path, "NEU-DET")
img_height = 224
img_width = 224
batch_size = 32

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    color_mode="grayscale",
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    color_mode="grayscale",
    image_size=(img_height, img_width),
    batch_size=batch_size
)

Found 1800 files belonging to 2 classes.
Using 1440 files for training.
Found 1800 files belonging to 2 classes.
Using 360 files for validation.


In [ ]:
def to_rgb(image, label):
    image = tf.image.grayscale_to_rgb(image)
    image = image / 255.0
    return image, label

train_ds = train_ds.map(to_rgb)
val_ds = val_ds.map(to_rgb)

In [ ]:
num_classes = 6

def build_model(base_model):
    base_model.trainable = False

    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(128, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
resnet_model = build_model(
    ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
)

history_resnet = resnet_model.fit(train_ds, validation_data=val_ds, epochs=5)
inception_model = build_model(
    InceptionV3(weights='imagenet', include_top=False, input_shape=(224,224,3))
)

history_inception = inception_model.fit(train_ds, validation_data=val_ds, epochs=5)
efficient_model = build_model(
    EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
)

history_efficient = efficient_model.fit(train_ds, validation_data=val_ds, epochs=5)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 266s 6s/step - accuracy: 0.7778 - loss: 0.5750 - val_accuracy: 0.8167 - val_loss: 0.5332
Epoch 2/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 224s 5s/step - accuracy: 0.7958 - loss: 0.5436 - val_accuracy: 0.8167 - val_loss: 0.4929
Epoch 3/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 254s 6s/step - accuracy: 0.7958 - loss: 0.5312 - val_accuracy: 0.8167 - val_loss: 0.4773
Epoch 4/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 257s 6s/step - accuracy: 0.7958 - loss: 0.5248 - val_accuracy: 0.8167 - val_loss: 0.4719
Epoch 5/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 220s 5s/step - accuracy: 0.7958 - loss: 0.5055 - val_accuracy: 0.8167 - val_loss: 0.4747
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Epoch 1/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 168s 4s/step - accuracy: 0.7750 - loss: 0.5723 - val_accuracy: 0.8333 - val_loss: 0.3995
Epoch 2/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 187s 3s/step - accuracy: 0.8139 - loss: 0.4239 - val_accuracy: 0.8389 - val_loss: 0.3725
Epoch 3/5
45/45 

In [ ]:
resnet_acc = resnet_model.evaluate(val_ds)[1]
inception_acc = inception_model.evaluate(val_ds)[1]
efficient_acc = efficient_model.evaluate(val_ds)[1]

print("ResNet50 Accuracy:", resnet_acc)
print("InceptionNet Accuracy:", inception_acc)
print("EfficientNet Accuracy:", efficient_acc)

12/12 ━━━━━━━━━━━━━━━━━━━━ 45s 4s/step - accuracy: 0.8167 - loss: 0.4747
12/12 ━━━━━━━━━━━━━━━━━━━━ 30s 2s/step - accuracy: 0.8417 - loss: 0.3700
12/12 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.8167 - loss: 0.5033
ResNet50 Accuracy: 0.8166666626930237
InceptionNet Accuracy: 0.8416666388511658
EfficientNet Accuracy: 0.8166666626930237
